# LEB5052 – Previsão de Chuva Diária na ESALQ/USP (1917–2024)

**Autor:** Arthur Cruvinel Marques  
**Instituição:** Departamento de Engenharia de Biossistemas, ESALQ/USP – Piracicaba, SP  
**Docente:** Prof. Dr. Roberto Fray da Silva

Este notebook reproduz integralmente as análises, tabelas e gráficos do artigo final:
1. Carregamento da base (`Dados_chuva_esalq.csv`) e imputação estritamente causal (*forward-fill*);
2. Engenharia de 12 atributos antecedentes e remoção das bordas temporais (39.166 dias válidos);
3. Análise exploratória por partição cronológica;
4. Modelagem comparativa (Majoritário, Persistência, Regressão Logística, Random Forest e HistGradientBoosting) com métricas de treino e teste;
5. Calibração de limiar na validação temporal (2015–2019) e aplicação no teste independente (2020–2024);
6. Importância por permutação no `HistGradientBoosting` e geração dos gráficos em alta resolução.

In [ ]:
# 1. Importacoes e configuracao
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({"font.size": 10, "figure.titlesize": 11})

def fmt_virgula(x, _pos=None, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")


In [ ]:
# 2. Carregamento dos dados (local ou direto do repositorio GitHub)
DATA_PATH = "Dados_chuva_esalq.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "https://raw.githubusercontent.com/percegd/previsao-de-chuva-ESALQ/main/Dados_chuva_esalq.csv"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.columns = ["dia_num", "ano", "precip_raw", "temp_raw"]
df["precip"] = pd.to_numeric(df["precip_raw"].astype(str).str.replace(",", "."), errors="coerce")
df["temp"] = pd.to_numeric(df["temp_raw"].astype(str).str.replace(",", "."), errors="coerce")

# Remove picos isolados de leitura de 1 dia (salto >= 10 C em relacao ao dia anterior e ao seguinte, ex.: 0 C em 1981)
picos_temp = ((df["temp"] - df["temp"].shift(1)).abs() >= 10) & ((df["temp"] - df["temp"].shift(-1)).abs() >= 10)
df.loc[picos_temp, "temp"] = np.nan

# Imputacao estritamente causal (forward-fill)
df["precip_imputed"] = df["precip"].ffill()
df["temp_imputed"] = df["temp"].ffill()

print(f"Registros brutos: {len(df)} | Faltantes precip: {df['precip'].isna().sum()} | Faltantes temp: {df['temp'].isna().sum()}")


In [ ]:
# 3. Engenharia de atributos e remocao das bordas temporais (7 iniciais + 1 final)
df["precip_amanha"] = df["precip_imputed"].shift(-1)
df = df.dropna(subset=["precip_amanha"]).copy()
df["chuva_amanha"] = (df["precip_amanha"] >= 0.1).astype(int)

for lag in [1, 2, 3, 7]:
    df[f"precip_lag_{lag}"] = df["precip_imputed"].shift(lag)
for lag in [1, 2]:
    df[f"temp_lag_{lag}"] = df["temp_imputed"].shift(lag)

df["precip_acum_7d"] = df["precip_imputed"].rolling(window=7, min_periods=1).sum()
df["temp_media_3d"] = df["temp_imputed"].rolling(window=3, min_periods=1).mean()
df["dia_do_ano"] = df["dia_num"]
df["mes"] = (((df["dia_num"] - 1) // 30) + 1).clip(1, 12)

df_model = df.drop(columns=["precip_raw", "temp_raw", "precip", "temp", "precip_amanha"]).dropna().copy()
print(f"Dias validos apos bordas temporais: {len(df_model)}")


In [ ]:
# 4. Analise exploratoria por particao cronologica
blocos = [
    ("Sub-treino (1917-2014)", df_model["ano"] <= 2014),
    ("Validacao (2015-2019)", (df_model["ano"] >= 2015) & (df_model["ano"] <= 2019)),
    ("Treino completo (1917-2019)", df_model["ano"] <= 2019),
    ("Teste independente (2020-2024)", df_model["ano"] > 2019),
]
eda = []
for nome, mask in blocos:
    sub = df_model[mask]
    eda.append({
        "Periodo": nome, "Dias": len(sub),
        "Dias chuvosos": int(sub["chuva_amanha"].sum()),
        "Taxa chuva": f"{sub['chuva_amanha'].mean()*100:.2f}%",
        "Precip. media (mm)": f"{sub['precip_imputed'].mean():.2f}",
        "Temp. media (C)": f"{sub['temp_imputed'].mean():.2f} +/- {sub['temp_imputed'].std():.2f}",
    })
pd.DataFrame(eda)


In [ ]:
# 5. Particao cronologica e treinamento dos 5 modelos
features = [
    "precip_imputed", "temp_imputed", "mes", "dia_do_ano",
    "precip_lag_1", "precip_lag_2", "precip_lag_3", "precip_lag_7",
    "temp_lag_1", "temp_lag_2", "precip_acum_7d", "temp_media_3d"
]
X, y = df_model[features], df_model["chuva_amanha"]
X_subtrain, y_subtrain = X[df_model["ano"] <= 2014], y[df_model["ano"] <= 2014]
X_val, y_val = X[(df_model["ano"] >= 2015) & (df_model["ano"] <= 2019)], y[(df_model["ano"] >= 2015) & (df_model["ano"] <= 2019)]
X_train, y_train = X[df_model["ano"] <= 2019], y[df_model["ano"] <= 2019]
X_test, y_test = X[df_model["ano"] > 2019], y[df_model["ano"] > 2019]

modelos = {}

t0 = time.time()
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
modelos["Majoritário (Naive)"] = (dummy.predict(X_train), dummy.predict_proba(X_train)[:, 1], dummy.predict(X_test), dummy.predict_proba(X_test)[:, 1], dummy, time.time() - t0)

t0 = time.time()
pred_tr_p = (X_train["precip_imputed"] >= 0.1).astype(int).values
pred_te_p = (X_test["precip_imputed"] >= 0.1).astype(int).values
modelos["Persistência (Naive)"] = (pred_tr_p, pred_tr_p.astype(float), pred_te_p, pred_te_p.astype(float), None, time.time() - t0)

scaler = StandardScaler()
X_tr_s, X_te_s = scaler.fit_transform(X_train), scaler.transform(X_test)
t0 = time.time()
lr = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr_s, y_train)
p_tr_lr, p_te_lr = lr.predict_proba(X_tr_s)[:, 1], lr.predict_proba(X_te_s)[:, 1]
modelos["Regressão Logística"] = ((p_tr_lr >= 0.5).astype(int), p_tr_lr, (p_te_lr >= 0.5).astype(int), p_te_lr, lr, time.time() - t0)

t0 = time.time()
rf = RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1).fit(X_train, y_train)
p_tr_rf, p_te_rf = rf.predict_proba(X_train)[:, 1], rf.predict_proba(X_test)[:, 1]
modelos["Random Forest"] = ((p_tr_rf >= 0.5).astype(int), p_tr_rf, (p_te_rf >= 0.5).astype(int), p_te_rf, rf, time.time() - t0)

t0 = time.time()
hgb = HistGradientBoostingClassifier(max_iter=150, max_depth=8, random_state=42).fit(X_train, y_train)
p_tr_hgb, p_te_hgb = hgb.predict_proba(X_train)[:, 1], hgb.predict_proba(X_test)[:, 1]
modelos["HistGradientBoosting"] = ((p_tr_hgb >= 0.5).astype(int), p_tr_hgb, (p_te_hgb >= 0.5).astype(int), p_te_hgb, hgb, time.time() - t0)

linhas = []
for nome, (_, prob_tr, pred_te, prob_te, _, tempo) in modelos.items():
    linhas.append({
        "Modelo": nome,
        "ROC-AUC Treino": f"{roc_auc_score(y_train, prob_tr):.4f}",
        "ROC-AUC Teste": f"{roc_auc_score(y_test, prob_te):.4f}",
        "Acuracia Teste": f"{accuracy_score(y_test, pred_te)*100:.2f}%",
        "Sensibilidade": f"{recall_score(y_test, pred_te)*100:.2f}%",
        "Precisao": f"{precision_score(y_test, pred_te, zero_division=0)*100:.2f}%",
        "F1-Score": f"{f1_score(y_test, pred_te, zero_division=0):.4f}",
        "Tempo Treino": f"{tempo:.2f} s" if tempo >= 0.01 else "< 0.01 s",
    })
pd.DataFrame(linhas)


In [ ]:
# 6. Calibracao de limiar na validacao (2015-2019) e aplicacao no teste (2020-2024)
hgb_val = HistGradientBoostingClassifier(max_iter=150, max_depth=8, random_state=42).fit(X_subtrain, y_subtrain)
prob_val = hgb_val.predict_proba(X_val)[:, 1]

calib = []
for th in [0.50, 0.40, 0.35, 0.30]:
    pred = (prob_val >= th).astype(int)
    calib.append({
        "Conjunto": "Validação (2015–2019)", "Limiar": f"{th:.2f}",
        "Acuracia": f"{accuracy_score(y_val, pred)*100:.2f}%",
        "Sensibilidade": f"{recall_score(y_val, pred)*100:.2f}%",
        "Precisao": f"{precision_score(y_val, pred)*100:.2f}%",
        "F1-Score": f"{f1_score(y_val, pred):.4f}",
    })

pred_te_035 = (p_te_hgb >= 0.35).astype(int)
calib.append({
    "Conjunto": "Teste (2020–2024)", "Limiar": "0.35 (aplicado)",
    "Acuracia": f"{accuracy_score(y_test, pred_te_035)*100:.2f}%",
    "Sensibilidade": f"{recall_score(y_test, pred_te_035)*100:.2f}%",
    "Precisao": f"{precision_score(y_test, pred_te_035)*100:.2f}%",
    "F1-Score": f"{f1_score(y_test, pred_te_035):.4f}",
})
pd.DataFrame(calib)


In [ ]:
# 7. Graficos finais (Curvas ROC, Matrizes de Confusao e Importancia por Permutacao - HistGradientBoosting)
modelos_prob = {k: v for k, v in modelos.items() if k not in ("Majoritário (Naive)", "Persistência (Naive)")}
cores = {"Regressão Logística": "darkorange", "Random Forest": "royalblue", "HistGradientBoosting": "forestgreen"}

fig, ax = plt.subplots(figsize=(7.2, 5.0))
for nome, (_, _, _, prob_te, _, _) in modelos_prob.items():
    fpr, tpr, _ = roc_curve(y_test, prob_te)
    auc = roc_auc_score(y_test, prob_te)
    ax.plot(fpr, tpr, label=f"{nome} (AUC = {auc:.3f})".replace(".", ","), color=cores[nome], lw=1.9)
tn, fp, fn, tp = confusion_matrix(y_test, pred_te_p).ravel()
ax.scatter([fp / (fp + tn)], [tp / (tp + fn)], color="#8e44ad", s=65, zorder=5,
           label=f"Persistência (AUC = {roc_auc_score(y_test, pred_te_p):.3f})".replace(".", ","))
ax.plot([0, 1], [0, 1], "k--", lw=1.1, label="Acaso (AUC = 0,500)")
ax.set_title("Curvas ROC no Período de Teste Independente (2020–2024)")
ax.set_xlabel("Taxa de Falsos Positivos")
ax.set_ylabel("Taxa de Verdadeiros Positivos")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: fmt_virgula(x, pos, 1)))
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: fmt_virgula(x, pos, 1)))
ax.legend(loc="lower right", frameon=True)
plt.tight_layout()
plt.show()

perm = permutation_importance(hgb, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=42, n_jobs=-1)
ordem = perm.importances_mean.argsort()
fig, ax = plt.subplots(figsize=(8.8, 5.2))
ax.barh(range(len(ordem)), perm.importances_mean[ordem], align="center", color="#2c3e50")
ax.set_yticks(range(len(ordem)))
ax.set_yticklabels(np.array(features)[ordem])
ax.set_title("Importância dos Atributos – HistGradientBoosting (Permutação no Teste)")
ax.set_xlabel("Queda média na ROC-AUC")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: fmt_virgula(x, pos, 2)))
ax.axvline(0, color="#7f8c8d", lw=0.8)
plt.tight_layout()
plt.show()
